# Extended Dueling DQN

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from models import DQN, DuelingDQN
from replay_buffer import ReplayBuffer
from train import ConfigDQN, actualizar_modelo, entrenar_dqn
from evaluation import evaluar_modelo

SEMILLA = 42
N_ACCIONES = 6

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Dispositivo seleccionado: {device}")
print(f"Número de acciones: {N_ACCIONES}")
print(f"Semilla: {SEMILLA}")

Dispositivo seleccionado: mps
Número de acciones: 6
Semilla: 42


In [3]:
config_extendido = ConfigDQN(
    nombre_experimento="v3_dueling_double_dqn_extendido",
    total_pasos=1_000_000,
)

resultado_extendido = entrenar_dqn(
    config=config_extendido,
    clase_modelo=DuelingDQN,
    device=device,
    usar_double_dqn=True,
    ruta_checkpoint_inicial=(
        "../models/v3_dueling_double_dqn/"
        "checkpoint_final.pt"
    ),
)

Reanudando entrenamiento desde el paso 500,000 (checkpoint: ../models/v3_dueling_double_dqn/checkpoint_final.pt)
Paso 501,000/1,000,000 | episodio=3 | epsilon=0.100 | loss=0.0034 | Q=1.967
Paso 502,000/1,000,000 | episodio=8 | epsilon=0.100 | loss=0.0030 | Q=1.691
Paso 503,000/1,000,000 | episodio=11 | epsilon=0.100 | loss=0.0018 | Q=1.703
Paso 504,000/1,000,000 | episodio=13 | epsilon=0.100 | loss=0.0243 | Q=1.849
Paso 505,000/1,000,000 | episodio=17 | epsilon=0.100 | loss=0.0016 | Q=1.694
Paso 506,000/1,000,000 | episodio=21 | epsilon=0.100 | loss=0.0036 | Q=1.585
Paso 507,000/1,000,000 | episodio=26 | epsilon=0.100 | loss=0.0013 | Q=1.421
Paso 508,000/1,000,000 | episodio=26 | epsilon=0.100 | loss=0.0041 | Q=1.954
Paso 509,000/1,000,000 | episodio=31 | epsilon=0.100 | loss=0.0067 | Q=1.628
Paso 510,000/1,000,000 | episodio=35 | epsilon=0.100 | loss=0.0138 | Q=1.125
Paso 511,000/1,000,000 | episodio=39 | epsilon=0.100 | loss=0.0293 | Q=1.632
Paso 512,000/1,000,000 | episodio=43 | eps

In [5]:
ruta_mejor_dueling_extendido = "../models/v3_dueling_double_dqn_extendido/mejor_modelo.pt"

from models import DuelingDQN

checkpoint_dueling_extendido = torch.load(
    ruta_mejor_dueling_extendido,
    map_location=device,
    weights_only=True,
)

mejor_dueling_dqn_extendido = DuelingDQN(
    n_acciones=N_ACCIONES
).to(device)

mejor_dueling_dqn_extendido.load_state_dict(
    checkpoint_dueling_extendido[
        "modelo_online_state_dict"
    ]
)

mejor_dueling_dqn_extendido.eval()

print(
    f"Checkpoint cargado desde el paso: "
    f"{checkpoint_dueling_extendido['paso']:,}"
)

Checkpoint cargado desde el paso: 850,000


In [6]:
N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42

print(
    "\nEvaluando Dueling Double DQN extendido durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

(
    resultados_dueling_extendido,
    resumen_dueling_extendido,
) = evaluar_modelo(
    modelo=mejor_dueling_dqn_extendido,
    config=config_extendido,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_dueling_dqn_extendido = pd.DataFrame(
    resultados_dueling_extendido
)

df_dueling_dqn_extendido["agente"] = (
    "Dueling Double DQN extendido"
)

df_dueling_dqn_extendido = (
    df_dueling_dqn_extendido[
        [
            "agente",
            "episodio",
            "seed",
            "recompensa_total",
            "pasos",
            "terminated",
            "truncated",
        ]
    ]
)

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_dueling_extendido.items():
    print(f"{metrica}: {valor:.2f}")

display(df_dueling_dqn_extendido.head(10))


Evaluando Dueling Double DQN extendido durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 446.83
mediana: 420.00
desviacion: 139.25
minimo: 175.00
maximo: 720.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,Dueling Double DQN extendido,0,42,335.0,587,True,False
1,Dueling Double DQN extendido,1,43,295.0,518,True,False
2,Dueling Double DQN extendido,2,44,530.0,1007,True,False
3,Dueling Double DQN extendido,3,45,370.0,589,True,False
4,Dueling Double DQN extendido,4,46,635.0,823,True,False
5,Dueling Double DQN extendido,5,47,545.0,947,True,False
6,Dueling Double DQN extendido,6,48,720.0,1022,True,False
7,Dueling Double DQN extendido,7,49,420.0,748,True,False
8,Dueling Double DQN extendido,8,50,420.0,671,True,False
9,Dueling Double DQN extendido,9,51,570.0,858,True,False
